# Fine-grained SGG Evaluation Analysis

读取 `eval_results.pytorch` 和 `result_dict.pytorch`，做谓词级召回、图像级失败案例、score 分布、relationness 影响等细粒度分析。

使用方式：先在下面配置 `EVAL_RESULTS_PATH` 和 `RESULT_DICT_PATH`。如果留空，notebook 会在仓库下自动搜索。

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'maskrcnn_benchmark').exists():
    REPO_ROOT = Path('/Users/shangfei/Developer/SDSGG')
sys.path.insert(0, str(REPO_ROOT))

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 80)
plt.rcParams['figure.figsize'] = (10, 4)

# 手动填路径；留空则自动搜索最新文件。
EVAL_RESULTS_PATH = ''
RESULT_DICT_PATH = ''

# 如果你想保存分析表格，填一个目录；留空则不保存。
ANALYSIS_OUTPUT_DIR = ''

In [ ]:
VG_REL_CLASSES = [
    '__background__', 'above', 'across', 'against', 'along', 'and', 'at', 'attached to', 'behind',
    'belonging to', 'between', 'carrying', 'covered in', 'covering', 'eating', 'flying in', 'for',
    'from', 'growing on', 'hanging from', 'has', 'holding', 'in', 'in front of', 'laying on',
    'looking at', 'lying on', 'made of', 'mounted on', 'near', 'of', 'on', 'on back of', 'over',
    'painted on', 'parked on', 'part of', 'playing', 'riding', 'says', 'sitting on', 'standing on',
    'to', 'under', 'using', 'walking in', 'walking on', 'watching', 'wearing', 'wears', 'with'
]

def rel_name(idx):
    idx = int(idx)
    if 0 <= idx < len(VG_REL_CLASSES):
        return VG_REL_CLASSES[idx]
    return f'rel_{idx}'

def to_np(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def has_field(box, name):
    return hasattr(box, 'has_field') and box.has_field(name)

def get_field_np(box, name, default=None):
    if not has_field(box, name):
        return default
    return to_np(box.get_field(name))

def box_fields(box):
    return sorted(list(getattr(box, 'extra_fields', {}).keys()))

def auto_find(name):
    candidates = list(REPO_ROOT.rglob(name))
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

def save_df(df, name):
    if not ANALYSIS_OUTPUT_DIR:
        return
    out_dir = Path(ANALYSIS_OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / name, index=False)
    print('saved', out_dir / name)

In [ ]:
eval_path = Path(EVAL_RESULTS_PATH) if EVAL_RESULTS_PATH else auto_find('eval_results.pytorch')
result_path = Path(RESULT_DICT_PATH) if RESULT_DICT_PATH else auto_find('result_dict.pytorch')
print('eval_results:', eval_path)
print('result_dict:', result_path)
assert eval_path and eval_path.exists(), '找不到 eval_results.pytorch，请手动填写 EVAL_RESULTS_PATH'
assert result_path and result_path.exists(), '找不到 result_dict.pytorch，请手动填写 RESULT_DICT_PATH'

eval_blob = torch.load(str(eval_path), map_location='cpu')
result_dict = torch.load(str(result_path), map_location='cpu')
groundtruths = eval_blob['groundtruths']
predictions = eval_blob['predictions']
print('num images:', len(predictions), 'num gt:', len(groundtruths))
print('prediction fields:', box_fields(predictions[0]))
print('groundtruth fields:', box_fields(groundtruths[0]))
print('result_dict keys:', sorted(result_dict.keys()))

## 1. 汇总指标与谓词级 mR

In [ ]:
summary_rows = []
for key, value in result_dict.items():
    if isinstance(value, dict) and set(value.keys()) >= {20, 50, 100}:
        row = {'metric': key}
        for k in [20, 50, 100]:
            v = value[k]
            if isinstance(v, list):
                row[f'@{k}'] = float(np.mean(v)) if len(v) else np.nan
            else:
                row[f'@{k}'] = float(v)
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
save_df(summary_df, 'metric_summary.csv')

In [ ]:
def predicate_recall_table(prefix='predcls_mean_recall_list'):
    rows = []
    if prefix not in result_dict:
        print('missing', prefix)
        return pd.DataFrame()
    for k in [20, 50, 100]:
        recalls = result_dict[prefix].get(k, [])
        for i, r in enumerate(recalls, start=1):
            rows.append({'rel_id': i, 'predicate': rel_name(i), 'K': k, 'recall': float(r)})
    return pd.DataFrame(rows)

mr_key = next((k for k in result_dict if k.endswith('_mean_recall_list')), 'predcls_mean_recall_list')
ng_mr_key = next((k for k in result_dict if k.endswith('_ng_mean_recall_list')), None)
mr_df = predicate_recall_table(mr_key)
pivot_mr = mr_df.pivot(index=['rel_id', 'predicate'], columns='K', values='recall').reset_index()
pivot_mr = pivot_mr.sort_values(100, ascending=True)
display(pivot_mr)
save_df(pivot_mr, 'predicate_mean_recall.csv')

ax = pivot_mr.tail(20).sort_values(100).plot.barh(x='predicate', y=100, legend=False, title='Top predicates by mR@100')
ax.set_xlabel('recall')
plt.show()
ax = pivot_mr.head(20).sort_values(100, ascending=False).plot.barh(x='predicate', y=100, legend=False, title='Worst predicates by mR@100')
ax.set_xlabel('recall')
plt.show()

## 2. 预测结果展开为 pair-level 表

In [ ]:
def build_prediction_table(topn_per_image=None):
    rows = []
    for image_idx, pred in enumerate(predictions):
        rel_pairs = get_field_np(pred, 'rel_pair_idxs')
        rel_scores = get_field_np(pred, 'pred_rel_scores')
        if rel_pairs is None or rel_scores is None or len(rel_pairs) == 0:
            continue
        pred_labels = get_field_np(pred, 'pred_labels')
        pred_obj_scores = get_field_np(pred, 'pred_scores')
        rel_logits = get_field_np(pred, 'pred_rel_logit')
        relationness = get_field_np(pred, 'relationness_scores')
        fg_scores = rel_scores[:, 1:]
        top_rel = fg_scores.argmax(axis=1) + 1
        top_score = fg_scores.max(axis=1)
        order = np.arange(len(rel_pairs))
        if topn_per_image is not None:
            order = order[:topn_per_image]
        for rank, j in enumerate(order):
            s, o = rel_pairs[j]
            row = {
                'image_idx': image_idx,
                'rank': rank,
                'sub_idx': int(s),
                'obj_idx': int(o),
                'sub_cls': int(pred_labels[s]) if pred_labels is not None else -1,
                'obj_cls': int(pred_labels[o]) if pred_labels is not None else -1,
                'sub_score': float(pred_obj_scores[s]) if pred_obj_scores is not None else np.nan,
                'obj_score': float(pred_obj_scores[o]) if pred_obj_scores is not None else np.nan,
                'pred_rel': int(top_rel[j]),
                'pred_rel_name': rel_name(top_rel[j]),
                'pred_rel_score': float(top_score[j]),
                'relationness': float(relationness[j]) if relationness is not None else np.nan,
            }
            if rel_logits is not None:
                row['pred_rel_logit_max'] = float(rel_logits[j, 1:].max())
                row['pred_rel_logit_bg'] = float(rel_logits[j, 0])
            rows.append(row)
    return pd.DataFrame(rows)

pred_df = build_prediction_table()
display(pred_df.head())
display(pred_df.describe(include='all'))
save_df(pred_df, 'prediction_pairs.csv')

In [ ]:
if len(pred_df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    pred_df['pred_rel_score'].hist(ax=axes[0], bins=50)
    axes[0].set_title('predicate max probability')
    if pred_df['relationness'].notna().any():
        pred_df['relationness'].hist(ax=axes[1], bins=50)
        axes[1].set_title('relationness')
        pred_df.plot.scatter(x='relationness', y='pred_rel_score', alpha=0.1, ax=axes[2])
        axes[2].set_title('relationness vs predicate score')
    plt.tight_layout()
    plt.show()

    pred_count = pred_df.groupby(['pred_rel', 'pred_rel_name']).size().reset_index(name='num_predictions')
    pred_count = pred_count.sort_values('num_predictions', ascending=False)
    display(pred_count.head(30))
    save_df(pred_count, 'predicate_prediction_count.csv')

## 3. GT pair 近似匹配分析

PredCls/SGCls 下，GT pair 的 subject/object index 通常可直接对齐。这里统计 GT pair 上 top-1 谓词是否正确，以及常见混淆。

In [ ]:
def build_gt_pair_table():
    rows = []
    for image_idx, (gt, pred) in enumerate(zip(groundtruths, predictions)):
        gt_rels = get_field_np(gt, 'relation_tuple')
        rel_pairs = get_field_np(pred, 'rel_pair_idxs')
        rel_scores = get_field_np(pred, 'pred_rel_scores')
        relationness = get_field_np(pred, 'relationness_scores')
        if gt_rels is None or rel_pairs is None or rel_scores is None:
            continue
        pair_to_pred = {tuple(map(int, p)): idx for idx, p in enumerate(rel_pairs.tolist())}
        for gt_idx, (s, o, r) in enumerate(gt_rels.astype(int).tolist()):
            j = pair_to_pred.get((s, o))
            if j is None:
                rows.append({
                    'image_idx': image_idx, 'gt_idx': gt_idx, 'sub_idx': s, 'obj_idx': o,
                    'gt_rel': r, 'gt_rel_name': rel_name(r), 'found_pair': False,
                    'pred_rel': -1, 'pred_rel_name': 'missing_pair', 'pred_rel_score': np.nan,
                    'gt_rel_score': np.nan, 'rank_of_gt_rel': np.nan, 'relationness': np.nan,
                    'correct_top1': False,
                })
                continue
            scores = rel_scores[j, 1:]
            pred_r = int(scores.argmax() + 1)
            gt_score = float(rel_scores[j, r]) if r < rel_scores.shape[1] else np.nan
            rank_gt = int((-scores).argsort().tolist().index(r - 1) + 1) if r > 0 and r <= scores.shape[0] else np.nan
            rows.append({
                'image_idx': image_idx, 'gt_idx': gt_idx, 'sub_idx': s, 'obj_idx': o,
                'gt_rel': r, 'gt_rel_name': rel_name(r), 'found_pair': True,
                'pred_rel': pred_r, 'pred_rel_name': rel_name(pred_r),
                'pred_rel_score': float(scores.max()), 'gt_rel_score': gt_score,
                'rank_of_gt_rel': rank_gt,
                'relationness': float(relationness[j]) if relationness is not None else np.nan,
                'correct_top1': pred_r == r,
            })
    return pd.DataFrame(rows)

gt_pair_df = build_gt_pair_table()
display(gt_pair_df.head())
print('GT pair top1 accuracy:', gt_pair_df['correct_top1'].mean() if len(gt_pair_df) else np.nan)
display(gt_pair_df.groupby(['gt_rel', 'gt_rel_name']).agg(
    count=('gt_rel', 'size'),
    top1_acc=('correct_top1', 'mean'),
    mean_gt_score=('gt_rel_score', 'mean'),
    mean_rank_gt=('rank_of_gt_rel', 'mean'),
    mean_relationness=('relationness', 'mean'),
).reset_index().sort_values('top1_acc').head(50))
save_df(gt_pair_df, 'gt_pair_top1_analysis.csv')

In [ ]:
if len(gt_pair_df):
    conf = gt_pair_df[gt_pair_df['found_pair']].groupby(
        ['gt_rel_name', 'pred_rel_name']
    ).size().reset_index(name='count').sort_values('count', ascending=False)
    wrong_conf = conf[conf['gt_rel_name'] != conf['pred_rel_name']]
    display(wrong_conf.head(50))
    save_df(wrong_conf, 'predicate_confusions.csv')

    if gt_pair_df['relationness'].notna().any():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        gt_pair_df.boxplot(column='relationness', by='correct_top1', ax=axes[0])
        axes[0].set_title('relationness by correctness')
        axes[0].figure.suptitle('')
        gt_pair_df.boxplot(column='gt_rel_score', by='correct_top1', ax=axes[1])
        axes[1].set_title('GT predicate score by correctness')
        axes[1].figure.suptitle('')
        plt.tight_layout()
        plt.show()

## 4. 图像级失败案例排序

In [ ]:
if len(gt_pair_df):
    image_df = gt_pair_df.groupby('image_idx').agg(
        num_gt=('gt_rel', 'size'),
        num_found_pair=('found_pair', 'sum'),
        top1_hits=('correct_top1', 'sum'),
        top1_acc=('correct_top1', 'mean'),
        mean_gt_score=('gt_rel_score', 'mean'),
        mean_relationness=('relationness', 'mean'),
    ).reset_index()
    image_df['miss_top1'] = image_df['num_gt'] - image_df['top1_hits']
    image_df = image_df.sort_values(['top1_acc', 'num_gt'], ascending=[True, False])
    display(image_df.head(50))
    save_df(image_df, 'image_failure_ranking.csv')

## 5. 单张图像展开查看

In [ ]:
IMAGE_IDX = int(image_df.iloc[0]['image_idx']) if 'image_df' in globals() and len(image_df) else 0
print('IMAGE_IDX =', IMAGE_IDX)

display(gt_pair_df[gt_pair_df['image_idx'] == IMAGE_IDX].sort_values(['correct_top1', 'rank_of_gt_rel']))
display(pred_df[pred_df['image_idx'] == IMAGE_IDX].head(100))